# Lecture 1 · Notebook 1 — The classic NLP pipeline
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture01/lecture01_1_classic_nlp_pipeline.ipynb)

**Companion to [Chapter 6 of the video lecture](https://nlp-course-cuhksz.github.io/lecture01.html#p6-s68)** (slide 68, *Pre-transformer era of NLP*) and to
[Chapter 5](https://nlp-course-cuhksz.github.io/lecture01.html#p5-s59) (slide 59, *Go beyond keyword matching*). Before large language models, an NLP system was a **stack of
separate components**: tokenization, part-of-speech tagging, parsing, named-entity recognition, coreference, and so on. In this
notebook you will:

1. see why plain **keyword matching** is not understanding,
2. run a real pipeline (spaCy) on the lecture's example *"Find me an Italian restaurant in New York City"*,
3. build a tiny **rule-based slot filler** on top of it, and watch it break on *"what's the weather there tomorrow?"*,
4. see **ambiguity** (*"I saw the man with the telescope"*) and **error propagation** down the stack.

Runs on CPU in about two minutes (the English model is ~13 MB).

In [ ]:
!python -m spacy download en_core_web_sm -q

In [ ]:
import spacy
from spacy import displacy
nlp = spacy.load("en_core_web_sm")
print("spaCy", spacy.__version__, "| pipeline components:", nlp.pipe_names)

## 1 · Keyword matching is the dog in the cartoon
Slide 59: we say a whole sentence, the dog hears *"blah blah Ginger blah blah"*. A keyword search engine does the same: it counts
shared words and ignores meaning. Here is a toy search engine over five documents.

In [ ]:
import re
docs = [
    "Low-cost airfare to France's capital this spring",
    "Paris Hilton launches a new perfume line",
    "How to keep your dog out of the garbage",
    "Cheap hotels near the Eiffel Tower",
    "Ginger: health benefits of a popular spice",
]
STOP = {"a", "an", "the", "to", "of", "out", "your", "my", "how", "this", "and", "in", "near", "s"}
def words(s):                                   # lower-cased words minus a few stop words
    return set(re.findall(r"[a-z]+", s.lower())) - STOP

def keyword_search(query):
    q = words(query)
    scores = [(len(q & words(d)), d) for d in docs]
    return sorted(scores, reverse=True)

for query in ["cheap flights to Paris", "Ginger, stay out of the trash!"]:
    print("query:", query)
    for s, d in keyword_search(query)[:3]:
        print(f"   {s} shared words | {d}")

The best answer to *"cheap flights to Paris"* (the airfare document) shares **no** word with the query, while the Paris Hilton story
and the hotels score on *Paris* and *cheap*. And the dog Ginger gets matched to the spice, not to the garbage advice. Real NLP has to get at structure and meaning, not surface
strings. (Lecture 2 shows how word vectors fix part of this.)

## 2 · The pipeline, stage by stage
spaCy runs the components listed above, one after the other. Each token gets a part-of-speech tag, a dependency label pointing to its
head word, and entity spans are marked on top.

In [ ]:
doc = nlp("Find me an Italian restaurant in New York City.")
print(f"{'token':12s} {'lemma':10s} {'POS':6s} {'dep':10s} head")
for t in doc:
    print(f"{t.text:12s} {t.lemma_:10s} {t.pos_:6s} {t.dep_:10s} {t.head.text}")
print("\nentities:", [(e.text, e.label_) for e in doc.ents])

The dependency parse as a picture (the syntactic tree on slide 68, drawn as arcs):

In [ ]:
displacy.render(doc, style="dep", jupyter=True, options={"compact": True, "distance": 95})

## 3 · A rule-based slot filler on top of the pipeline
Dialogue systems of the pre-Transformer era turned such analyses into a **frame**: an action plus slots. We write the rules by hand:
the root verb is the action, a nationality adjective (`NORP`) is the cuisine, a place (`GPE`) is the location, a `DATE` is the time.

In [ ]:
def fill_slots(text, state=None):
    doc = nlp(text)
    frame = {"action": None, "cuisine": None, "location": None, "time": None}
    for t in doc:
        if t.dep_ == "ROOT":
            frame["action"] = t.lemma_.lower()
    for e in doc.ents:
        if e.label_ == "NORP": frame["cuisine"] = e.text
        if e.label_ in ("GPE", "LOC"): frame["location"] = e.text
        if e.label_ == "DATE": frame["time"] = e.text
    return frame

print(fill_slots("Find me an Italian restaurant in New York City."))
print(fill_slots("What's the weather there tomorrow?"))

The second request has no location: *"there"* refers back to New York City, but nothing in this stack knows that. (Notice also the
action: the root of *"What's the weather"* is the verb *be*, so our "root verb = action" rule already gives nonsense.) A classic system needs
yet another component (**coreference resolution** or a **dialogue-state tracker**) that carries the location over from the previous
turn. Here is the simplest possible tracker, one more hand-written rule:

In [ ]:
def track(turns):
    state = {}
    for text in turns:
        frame = fill_slots(text)
        if frame["location"] is None and any(t.lower_ in ("there", "here") for t in nlp(text)):
            frame["location"] = state.get("location")       # resolve "there" by copying the last location
        state.update({k: v for k, v in frame.items() if v})
        print(f"{text:50s} -> {frame}")

track(["Find me an Italian restaurant in New York City.", "What's the weather there tomorrow?"])

## 4 · Ambiguity: the parser must pick one reading
Slide 57: *"I saw the man with the telescope"* has two meanings. Did I use the telescope, or did the man have it? A parser outputs one tree,
so it silently commits to one reading. Look at which word **with** attaches to.

In [ ]:
for s in ["I saw the man with the telescope.", "I saw the man with the hat.", "I ate the pizza with a fork."]:
    d = nlp(s)
    w = [t for t in d if t.lower_ == "with"][0]
    reading = "the seeing/eating (verb)" if w.head.pos_ == "VERB" else f"the {w.head.text} (noun)"
    print(f"{s:36s} 'with' attaches to '{w.head.text}'  -> modifies {reading}")

Only world knowledge tells you that a hat is worn and a fork is used. A small statistical parser has little of that; large language models
have much more, which is one reason they replaced these stacks.

## 5 · Errors propagate down the stack
Every stage depends on the one before it. Type the same request the way people actually type, in lower case and then with typos, and watch
what the entity recognizer, and therefore the slot filler, does.

In [ ]:
for text in ["Find me an Italian restaurant in New York City.",
             "find me an italian restaurant in new york city",
             "find me an itallian restaurant in new yrok"]:
    d = nlp(text)
    print(f"{text:48s} entities={[(e.text, e.label_) for e in d.ents]}")
    print(f"{'':48s} slots={fill_slots(text)}")

With typos the place is no longer recognized as an entity, so the location slot is empty, and every later stage (the dialogue tracker,
the restaurant database query) inherits the mistake. This is the weakness the lecture points out on slide 68: each stage is a separate
model, and **errors add up** as you go down the stack.
Notebook 3 gives the same requests to a single small LLM instead.

## Exercises
1. Extend `fill_slots` with a `price` slot (*cheap*, *expensive*, *under 50 dollars*). How many rules do you need before it feels robust?
2. Find a sentence of your own where the parser picks the wrong attachment or the wrong entity type. Which stage made the mistake?
3. Add a third turn, *"Book a table there for two at 7pm"*. What breaks, and which extra component would a classic system add for it?

**Next:** [Notebook 2 — what is a language model?]({colab(N2)})